In [14]:
import pandas as pd
import numpy as np
from utils import cma, portcon

# 1. Data Loading & Preprocessing

In [2]:
df_raw = cma.load_returns('spd_case_study_data_2026.xlsx')
excess_df_raw = cma.to_excess(df_raw)
excess_df,_ = cma.backfill_hy(excess_df_raw)
excess_df

,US equity,Developed ex-US Equity,Emerging Markets Equity,US Treasuries,US IG Credit,US HY Credit
2003-05-01,0.054032,0.063849,0.089060,0.035797,0.037072,0.007842
2003-06-01,0.006157,0.018437,0.041712,-0.006443,-0.000992,0.019057
2003-07-01,0.020965,0.014428,0.056296,-0.055448,-0.058589,-0.019568
2003-08-01,0.019851,0.032201,0.078507,0.005254,0.012801,0.015891
2003-09-01,-0.015459,0.029659,-0.006902,0.048423,0.044228,0.025065
...,...,...,...,...,...,...
2025-12-01,-0.005092,0.005827,0.004339,-0.013639,-0.014234,-0.002305
2026-01-01,0.014814,0.065139,0.092734,0.000994,0.008214,0.012524
2026-02-01,-0.011603,0.043108,0.055932,0.018480,0.007057,-0.007856
2026-03-01,-0.054924,-0.081248,-0.095481,-0.025802,-0.023414,-0.012438


In [3]:
excess_df.corr()

,US equity,Developed ex-US Equity,Emerging Markets Equity,US Treasuries,US IG Credit,US HY Credit
US equity,1.000000,0.851425,0.737295,-0.063078,0.417868,0.721821
Developed ex-US Equity,0.851425,1.000000,0.851727,0.017963,0.475662,0.735185
Emerging Markets Equity,0.737295,0.851727,1.000000,-0.017333,0.404391,0.669552
US Treasuries,-0.063078,0.017963,-0.017333,1.000000,0.685069,0.124870
US IG Credit,0.417868,0.475662,0.404391,0.685069,1.000000,0.673594
US HY Credit,0.721821,0.735185,0.669552,0.124870,0.673594,1.000000


# 2. CMA

In [4]:
mu, cov = cma.build_cmas(excess_df)

## 2.1 Expected Returns

In [10]:
mu

US equity                  0.102385
Developed ex-US Equity     0.074437
Emerging Markets Equity    0.098433
US Treasuries              0.018533
US IG Credit               0.027420
US HY Credit               0.038322
Name: Historical, dtype: float64

## 2.2 Covariance

In [11]:
cov

,US equity,Developed ex-US Equity,Emerging Markets Equity,US Treasuries,US IG Credit,US HY Credit
US equity,0.021520,0.020568,0.022036,-0.000604,0.004803,0.010064
Developed ex-US Equity,0.020568,0.028149,0.029187,0.000197,0.006269,0.011752
Emerging Markets Equity,0.022036,0.029187,0.042848,-0.000236,0.006593,0.013242
US Treasuries,-0.000604,0.000197,-0.000236,0.004809,0.003581,0.000792
US IG Credit,0.004803,0.006269,0.006593,0.003581,0.006721,0.005123
US HY Credit,0.010064,0.011752,0.013242,0.000792,0.005123,0.009677


In [58]:
d = cov.values.diagonal() ** 0.5
scaler = np.outer(d,d)
(cov/scaler).round(2)

,US equity,Developed ex-US Equity,Emerging Markets Equity,US Treasuries,US IG Credit,US HY Credit
US equity,1.00,0.84,0.73,-0.06,0.40,0.70
Developed ex-US Equity,0.84,1.00,0.84,0.02,0.46,0.71
Emerging Markets Equity,0.73,0.84,1.00,-0.02,0.39,0.65
US Treasuries,-0.06,0.02,-0.02,1.00,0.63,0.12
US IG Credit,0.40,0.46,0.39,0.63,1.00,0.64
US HY Credit,0.70,0.71,0.65,0.12,0.64,1.00


# 3. Portfolio Construction

In [16]:
cons = portcon.Constraints()
aa = portcon.construct_all(mu, cov, excess_df, cons)
aa

,US equity,Developed ex-US Equity,Emerging Markets Equity,US Treasuries,US IG Credit,US HY Credit
Equal Weight,0.166667,0.166667,0.166667,0.166667,0.166667,0.166667
Inverse Vol,0.125327,0.109581,0.088819,0.265114,0.224258,0.186900
Min Variance,0.200000,0.000000,0.000000,0.350000,0.345429,0.104571
Risk Parity,0.166524,0.166367,0.166206,0.167173,0.166938,0.166791
Max Sharpe,0.350000,0.000000,0.088860,0.350000,0.192140,0.019000
CVaR-Min,0.200000,0.000000,0.000000,0.350000,0.350000,0.100000


# Judgement & Ensemble